# RAG Pipeline Simple Project
## Steps Involved:
- Load PDF (PyPDFLoader reads the file) $\rightarrow$ Pages (Document objects with text and metadata)

- Split into Chunks (RecursiveCharacterTextSplitter cuts pages) $\rightarrow$ Chunks (Smaller, managed 500-character segments)

- Embed & Store (Open AI Embeddings convert text to vectors; FAISS vector store saves them) $\rightarrow$ Vector Database

- Retrieve (User Query enters) $\rightarrow$ Retriever (Finds top 3 most relevant chunks in FAISS) $\rightarrow$ Retrieved Chunks

- Generate Answer (Retrieved Chunks $\rightarrow$ Prompt Context, User Query $\rightarrow$ Prompt Question) $\rightarrow$ LLM (ChatGPT) $\rightarrow$ Final Answer

### Dependencies and Setup
- from langchain_openai import ChatOpenAI: The bridge that lets LangChain talk to ChatGpt for both generating text and creating embeddings.

- langchain-community: Contains third-party integrations like the PDF loader and the FAISS vector store.

- langchain-text-splitters: Specialized tools to break large documents into smaller, manageable pieces.

- faiss-cpu: A library by Meta for efficient similarity search. It acts as your "database" for text.

- pypdf: The engine that actually opens and reads the characters inside a PDF file.

In [ ]:
!pip install langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf -q

In [ ]:

from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
import os

# You must set your OpenAI key here
os.environ["OPENAI_API_KEY"] = "*************"

### Document Loading
PyPDFLoader: A class that takes a file path and prepares it for reading.

.load(): This function converts the PDF into a list of Document objects. Each object contains the text of one page and metadata (like the page number).

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "/content/python-handbook flavio.pdf"
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"Loaded {len(pages)} pages.")
print(f"Page 1 preview: {pages[0].page_content[:300]}...")

Loaded 93 pages.
Page 1 preview: ...


## Text Chunking
LLMs have a "context window" (a limit on how much text they can read at once). Also, embedding a whole book as one vector loses detail. We must chop the text into "chunks."

- RecursiveCharacterTextSplitter: This is the "smart" way to split text. It tries to split by paragraphs first, then sentences, then words, to ensure meanings aren't cut in half.

- chunk_overlap: We keep 50 characters from the end of Chunk A at the start of Chunk B. This ensures that if a vital piece of information is at the cut-off point, it’s captured in context in at least one chunk.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Max characters per chunk
    chunk_overlap=50,     # Shared characters between chunks
)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks.")
print(f"Chunk 0: {chunks[0].page_content[:200]}...")

Split into 199 chunks.
Chunk 0: 1
Table of Contents
Preface
The Python Handbook
Conclusion...


## Embeddings and Vector Storage
Computers don't understand words; they understand numbers. Embeddings convert text into a list of numbers (a vector) representing its meaning.

- OpenAIEmbeddings: Uses Open AI specialized model to turn your text chunks into vectors.

- FAISS.from_documents: This takes your text chunks, runs them through the embedding model, and stores them in a FAISS index. FAISS allows you to mathematically calculate which chunks are "closest" in meaning to a user's question.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS


embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"Vector store ready with {vectorstore.index.ntotal} vectors.")

Vector store ready with 199 vectors.


## Retrieval Chain
This is where the "R" in RAG happens. We need a way to pull the right chunks when a user asks a question.

- as_retriever: Converts the vector store into an interface that can take a string (query) and return documents.

- k=3: Tells the system to grab only the top 3 most relevant chunks.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
test_results = retriever.invoke("What is this document about?")
for i, doc in enumerate(test_results):
    print(f"--- Result {i+1} (page {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:200])
    print()

--- Result 1 (page 78) ---
to adhere to your favorite one.
I like Google's standard: https://github.com/google/styleguide/blob/gh-
pages/pyguide.md#38-comments-and-docstrings
Standard allows to have tools to extract docstrings 

--- Result 2 (page 1) ---
1
Table of Contents
Preface
The Python Handbook
Conclusion

--- Result 3 (page 78) ---
provided in form of docstrings.
Then, you can use print() to get information about a function:



## The Generation Chain
Now we set up Gemini to answer the question using the retrieved chunks.
- ChatGoogleGenerativeAI: This initializes the Gemini model. temperature=0 makes the AI predictable and factual (essential for RAG).

- ChatPromptTemplate: This is the "instruction" for the AI. It tells Gemini: "Don't use your general knowledge; only use the text I give you in the {context}."

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("""
You are a helpful and expert assistant. You have been given specific information from a document to help the user.

Instructions:
1. Use the provided context to answer the user's question directly and naturally.
2. DO NOT mention phrases like "based on the context" or "according to the document." Just state the facts.
3. Use multiple lines, headers, or bullet points to make the answer easy to read.
4. If the information is not in the context, politely say that you don't have that specific information yet.

Context:
{context}

Question: {input}
""")
#{context} → usually retrieved data (e.g., from documents, PDFs, database)
#{input} → user’s question

## Connecting Retriever and LLM Together.
1) create_stuff_documents_chain: This "stuffs" all retrieved chunks into the {context} part of your prompt.

2) create_retrieval_chain: This creates the final pipeline. When you call .invoke():

- It embeds your question.

- It finds the 3 best chunks in FAISS.

- It sends those chunks + your question to Gemini.

- Gemini generates the final answer.

### Asking Questions

In [ ]:
from langchain_community.vectorstores.pgembedding import QueryResult
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

response = rag_chain.invoke({"input": "Summarize the document"})
print("Question:",response["input"])
print()
print("Answer:",response["answer"])
print("\n--- Source chunks used ---")
for i, doc in enumerate(response["context"]):
    print(f"  Chunk {i+1} | page {doc.metadata.get('page', '?')} | {doc.page_content[:100]}...")

Question: Summarize the document

Answer: Here’s a summary of the key points from the document:

### Code Formatting
- **Indentation**: Use spaces instead of tabs for indentation.

### Comments and Docstrings
- **Comments**: Use comments to explain code, e.g., `num = 1  # this is another comment`.
- **Docstrings**: 
  - Follow conventions for consistency and automatic processing.
  - Example of defining a docstring for a function is provided.
  - Tools can extract docstrings to generate documentation.

### Style Guides
- A recommended style guide is Google's Python Style Guide, which can be found [here](https://github.com/google/styleguide/blob/gh-pages/pyguide.md#38-comments-and-docstrings).

### Introspection
- Functions, variables, and objects can be analyzed using introspection.
- The `help()` function can be used to access documentation provided in docstrings.

This summary captures the essential points regarding code formatting, documentation practices, and introspection in Pytho

In [ ]:
query = "Tell me python use cases in bullet points."


response = rag_chain.invoke({"input": query})

print("Answer:")
print(response["answer"])

print("\n--- Source chunks used ---")
for i, doc in enumerate(response["context"]):
    print(f"  Chunk {i+1} | page {doc.metadata.get('page', '?')} | {doc.page_content[:100]}...")

Answer:
- Shell scripting
- Task automation
- Web development
- Data analysis
- Machine learning
- Game development
- Working with embedded devices

--- Source chunks used ---
  Chunk 1 | page 5 | 5
Python is literally eating the programming world. It is growing in popularity
and usage in ways th...
  Chunk 2 | page 3 | 3
The Python Handbook
1. Introduction to Python
2. Installing Python
3. Running Python programs
4. P...
  Chunk 3 | page 4 | 38. Exceptions
39. The  with  statement
40. Installing 3rd party packages using  pip 
41. List compr...


In [ ]:
query = "Who is Bibek Subedi"


response = rag_chain.invoke({"input": query})

print("Answer:")
print(response["answer"])

print("\n--- Source chunks used ---")
for i, doc in enumerate(response["context"]):
    print(f"  Chunk {i+1} | page {doc.metadata.get('page', '?')} | {doc.page_content[:100]}...")

Answer:
I don't have that specific information yet.

--- Source chunks used ---
  Chunk 1 | page 1 | 1
Table of Contents
Preface
The Python Handbook
Conclusion...
  Chunk 2 | page 71 | Click and Python Prompt Toolkit.
30. Lambda Functions...
  Chunk 3 | page 4 | 38. Exceptions
39. The  with  statement
40. Installing 3rd party packages using  pip 
41. List compr...


In [ ]:
query = "Tuples in python with code"


response = rag_chain.invoke({"input": query})

print("Answer:")
print(response["answer"])

print("\n--- Source chunks used ---")
for i, doc in enumerate(response["context"]):
    print(f"  Chunk {i+1} | page {doc.metadata.get('page', '?')} | {doc.page_content[:100]}...")

Answer:
### Tuples in Python

- **Definition**: Tuples are immutable groups of objects, meaning once created, they cannot be modified (no adding or removing items).

- **Creation**: Tuples are created using parentheses `()` instead of square brackets `[]`.

  ```python
  names = ("Roger", "Syd")
  ```

- **Accessing Values**: Tuples are ordered, so you can access their values using an index.

  ```python
  first_name = names[0]  # Accesses "Roger"
  second_name = names[1]  # Accesses "Syd"
  ```

- **Creating New Tuples**: You can create a new tuple by concatenating existing tuples using the `+` operator.

  ```python
  newTuple = names + ("Vanille", "Tina")
  ```

This is how tuples work in Python!

--- Source chunks used ---
  Chunk 1 | page 48 | 48
that will return a new list, sorted, instead of modifying the original list.
17. Tuples
Tuples ar...
  Chunk 2 | page 52 | 52
19. Sets
Sets are another important Python data structure.
We can say they work like tuples, but ...
  Chunk 3 |

## Running it like a chatbot.

In [ ]:
import sys

# Aesthetic Header
print("="*50)
print("📚 PDF AI ASSISTANT ACTIVATED")
print("Ready to answer questions about your document.")
print("Type 'quit' to exit.")
print("="*50)

while True:
    # Adding extra space before the prompt for clarity
    print("\n" + "-"*30)
    user_query = input("❓ YOUR QUESTION: ")

    # Exit check
    if user_query.lower() in ['quit', 'exit', 'q']:
        print("\n👋 System: Shutting down. Have a great day!")
        print("="*50)
        break

    # Empty input check
    if not user_query.strip():
        print("⚠️ System: Please enter a valid question.")
        continue

    try:
        print("\n🔍 System: Searching document and generating answer...")

        # Calling the RAG chain
        response = rag_chain.invoke({"input": user_query})

        # Printing the Answer in a clean, multi-line format
        print("\n💡 ANSWER:")
        print("-" * 10)
        print(response['answer'])
        print("-" * 10)

    except Exception as e:
        print(f"\n❌ ERROR: Something went wrong.\nDetails: {e}")

📚 PDF AI ASSISTANT ACTIVATED
Ready to answer questions about your document.
Type 'quit' to exit.

------------------------------
❓ YOUR QUESTION: File handling in python

🔍 System: Searching document and generating answer...

💡 ANSWER:
----------
### File Handling in Python

- **Using `with` Statement**:
  - The `with` statement simplifies file handling by automatically managing resources.
  - It ensures that files are properly closed after their suite finishes, even if an error occurs.
  
  ```python
  filename = '/Users/flavio/test.txt' 
  with open(filename, 'r') as file: 
      content = file.read() 
      print(content) 
  ```

- **Benefits**:
  - Implicit exception handling: The `close()` method is called automatically.
  - Cleaner and more readable code.

- **General Practices**:
  - Always use the `with` statement for file operations to avoid resource leaks.

This approach is not limited to file handling; it can be applied to other resources as well.
----------

---------------

### I had used gemini free tier api but it exceeded its tier. So I used Chatgpt paid api key. There maybe some mismatch in the comment or text file. I revised the code.